# 03: Transformer & LLM Mechanics

*What's actually happening inside the black box you're extracting signals from.*

This notebook explains the machinery behind the model outputs we later analyze statistically: tokens, hidden states, attention, logits, sampling, and generation.

The overall theme is simple:

- text is first converted into tokens
- tokens are mapped into vectors
- the model updates those vectors using attention and feed-forward layers
- it produces logits over the vocabulary
- probabilities are derived from those logits
- the model generates next tokens one at a time

The most important practical fact for this project is this:

> We are not measuring "confidence in a word" in the abstract. We are measuring confidence over a token sequence under a model distribution.

## 03a. Tokenization (BPE)

Language models do not read raw text directly. They read tokens.

A token is a unit of text the model understands. It may be:

- a word fragment
- a common subword piece
- punctuation
- a special token such as BOS, EOS, or PAD

### Why tokenization matters

Text is messy and variable-length. But the model works with a fixed vocabulary and fixed-size vector representations.

So the pipeline is:

$$
\text{string} \rightarrow \text{token IDs} \rightarrow \text{embeddings} \rightarrow \text{hidden states} \rightarrow \text{logits}
$$

### BPE intuition

Byte Pair Encoding (BPE) starts from small pieces and repeatedly merges the most frequent pairs.

Example idea:

- "low" and "er" may be frequent enough to merge into "lower"
- "ing" may appear often enough to become one token
- unusual words can be split into familiar subpieces

This helps balance:

- vocabulary size
- representation of rare words
- ability to generalize to unseen morphology

### Why a word-level log-probability is really a token-sequence log-probability

Suppose the model sees:

"The capital of France is Paris."

The tokenizer may produce something like:

["The", " capital", " of", " France", " is", " Paris", "."]

or even more granular pieces such as:

["The", " cap", "ital", " of", " France", " is", " Paris", "."]

The model assigns a probability to each next token conditioned on the previous tokens:

$$
P(t_1, t_2, \dots, t_n) = \prod_{i=1}^{n} P(t_i \mid t_1, \dots, t_{i-1})
$$

So the log-probability of a sentence is:

$$
\log P(\text{sentence}) = \sum_i \log P(t_i \mid t_{<i})
$$

This is why in practice, when people say "the model's log-probability of the word," they are usually really talking about the log-probability of the token sequence that realizes that word.

### A toy example

If the tokenization of a phrase is:

["Paris", " is", " the", " capital", " of", " France"]

then the model does not assign one probability to the whole phrase as an atomic unit. It assigns probabilities token by token.

This matters substantially for entropy, confidence, and hallucination detection:

- some words are represented by several tokens
- tokenization affects the metric we compute
- longer tokenizations can lower or raise measured confidence in ways that are partly about segmentation, not meaning

### Implementation note

A tokenizer can be seen as a deterministic function:

$$
\text{tokenize}(\text{text}) = [123, 456, 789, \dots]
$$

and a decoder can convert the token IDs back to text.

The exact tokenization is a design choice of the model, not a property of the language itself.

In [4]:
from transformers import AutoTokenizer

# Example tokenization using a public model tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The capital of France is Paris."
tokens = tokenizer(text, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print(tokens["input_ids"][0])

['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'ĠParis', '.']
tensor([ 464, 3139,  286, 4881,  318, 6342,   13])


## 03b. The forward pass

The forward pass is the model transforming input tokens into output probabilities.

At a high level, the pipeline looks like this:

1. Token IDs become embeddings
2. Positional encodings are added
3. Repeated transformer blocks process the sequence
4. Attention mixes information across tokens
5. The final hidden states are projected to logits over the vocabulary

### Step 1: embeddings

Each token has a learned vector representation.

If a token ID is mapped to an embedding vector $e_i$, then the model starts with a sequence of vectors:

$$
E = [e_1, e_2, \dots, e_n]
$$

This is the initial semantic space the model operates in.

### Step 2: positional information

Since transformers do not naturally encode order by recurrence, they add positional information:

$$
H_0 = E + P
$$

where $P$ contains positional encodings.

This tells the model where each token sits in the sequence.

### Step 3: attention

Attention is the central operation in a transformer.

For each token, the model asks:

> Which other tokens in the sequence are relevant to this one?

The idea is not that it is reading a sentence in a handcrafted way; it is learning a weighted interaction pattern between tokens.

A simplified view is:

$$
\text{attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Conceptually:

- $Q$ = what the current token is looking for
- $K$ = what information is available
- $V$ = the actual content being passed along

This makes each token a context-aware representation that includes relevant preceding and sometimes following tokens, depending on the model architecture.

### Step 4: residual streams and feed-forward layers

Each transformer block usually does something like:

- attention to gather context
- residual connection to preserve previous information
- feed-forward neural network to enrich the representation
- another residual connection

This produces richer hidden states at each layer.

The important research fact is:

- early layers often encode syntax and local structure
- later layers become more task- and semantics-aware
- hidden states can be extracted and analyzed as signals

### Step 5: logits

At the end of the network, the final hidden state for a token position is projected into a vocabulary-sized vector:

$$
\ell = W h + b
$$

where:

- $h$ is the final hidden state
- $W$ is the output projection matrix
- $\ell$ is the logit vector

This vector is not yet a probability distribution. It is a score for each token in the vocabulary.

### Why this is important for uncertainty work

The model's uncertainty is not mysterious. It is encoded directly in how peaked or spread out the logits are before softmax. If the logits are sharp, the model is confident. If they are flat, it is uncertain.

## 03c. From logits to probabilities

The logits are transformed into probabilities via softmax.

For a logit vector $z = [z_1, z_2, \dots, z_V]$, the probability of token $i$ is:

$$
P(i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

where $T$ is the temperature.

### Softmax

When $T=1$, this is the standard softmax:

$$
P(i) = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

- higher logits become larger probabilities
- the outputs sum to 1
- this gives a proper probability distribution over the vocabulary

### Temperature scaling

Temperature changes the sharpness of the distribution:

- low temperature $T < 1$: sharper, more confident distributions
- high temperature $T > 1$: flatter, more spread-out distributions

This is useful for sampling and calibration.

### Top-k sampling

Top-k restricts sampling to the $k$ highest-probability tokens.

Example:

- sort the vocabulary probabilities
- keep only the top $k$ candidates
- renormalize over those candidates

This prevents the model from sampling from extremely unlikely tokens.

### Top-p (nucleus) sampling

Top-p keeps the smallest set of tokens whose cumulative probability exceeds $p$.

Example:

- rank tokens by probability
- keep tokens until their total probability reaches $p$
- sample from that subset

This adapts to the uncertainty in the distribution. If the model is certain, only a few tokens may be kept. If uncertain, more tokens may remain available.

### Why this reproduces the Entropy Lab math

Entropy and confidence are computed from the probability distribution over next tokens.

If we define a distribution over the vocabulary:

$$
q = [q_1, q_2, \dots, q_V]
$$

then:

$$
H(q) = -\sum_i q_i \log_2 q_i
$$

A very peaked distribution has low entropy. A flat distribution has high entropy.

This is exactly the kind of quantity we were earlier computing for toy distributions in the probability notebook.

The model does not do anything magical here: it just produces a real probability distribution over token IDs, and we compute entropy over that distribution.

In [5]:
import numpy as np

# Softmax definition

def softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=float) / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

# Example logits from a model
logits = np.array([2.0, 1.5, 0.3, -1.0])
probs = softmax(logits)
print("Softmax probabilities:", probs)
print("Sum:", probs.sum())

# Temperature effect
for temp in [0.5, 1.0, 2.0]:
    p = softmax(logits, temperature=temp)
    print(f"Temperature {temp}: {p}")

# Entropy of distribution

def entropy_from_probs(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

print("Entropy of this next-token distribution:", entropy_from_probs(probs))

# Top-k toy logic
k = 2
sorted_idx = np.argsort(probs)[::-1]
keep = sorted_idx[:k]
print("Top-k indices:", keep)

# Top-p toy logic
p_cutoff = 0.8
sorted_idx = np.argsort(probs)[::-1]
probs_sorted = probs[sorted_idx]
cdf = np.cumsum(probs_sorted)
keep_p = sorted_idx[cdf <= p_cutoff]
print("Top-p kept indices before cutoff:", keep_p)

Softmax probabilities: [0.54377342 0.32981525 0.09933844 0.02707288]
Sum: 1.0
Temperature 0.5: [0.71238697 0.26207252 0.02377468 0.00176583]
Temperature 1.0: [0.54377342 0.32981525 0.09933844 0.02707288]
Temperature 2.0: [0.41163344 0.32058045 0.17593828 0.09184784]
Entropy of this next-token distribution: 1.4776432623893798
Top-k indices: [0 1]
Top-p kept indices before cutoff: [0]


## 03d. The autoregressive generation loop

Autoregressive generation is the process of producing one token at a time, each time conditioning on the tokens already generated.

The model is trained to predict the next token given the previous context:

$$
P(x_{t+1} \mid x_1, x_2, \dots, x_t)
$$

### Teacher forcing

During training, the model is fed the ground-truth previous tokens.

This means at each step, it sees the correct context and is asked to predict the next token.

So the training objective is:

$$
\sum_t \log P(x_{t+1} \mid x_1, \dots, x_t)
$$

This gives a clean supervised signal and is computationally efficient.

### Free-running generation

At inference time, the model usually generates from its own previous outputs.

This matters because:

- generation errors compound over time
- the model may drift into a region of the distribution it was not originally conditioned on
- confidence estimates for generated text can look very different from teacher-forced evaluation

So:

- teacher forcing measures how well the model predicts the observed text
- free-running generation measures how the model behaves when it is generating its own continuation

These are not the same statistic.

### Why this matters for LLM evaluation

If you compute log-probability on a fixed reference text, you are often doing teacher-forced evaluation. But if you ask the model to generate freely, the actual behavior depends on sampling and decode strategy.

This is crucial for:

- hallucination scoring
- self-consistency checks
- uncertainty estimation
- semantic entropy pipelines

A model may look highly confident under teacher forcing but behave very differently when the tokens it generated earlier change future context.

In [6]:
# Toy autoregressive generation idea

# Sequence probability = product of next-token probabilities
seq = [0.8, 0.7, 0.9, 0.6]
logp = np.log(seq)
print("Log-probabilities for tokens:", logp)
print("Total sequence log-probability:", logp.sum())
print("Total sequence probability:", np.exp(logp.sum()))

# Notice: this is not a single probability for the whole phrase.
# It is the product of a sequence of conditional probabilities.

Log-probabilities for tokens: [-0.22314355 -0.35667494 -0.10536052 -0.51082562]
Total sequence log-probability: -1.1960046346767592
Total sequence probability: 0.30239999999999995


## 03e. PyTorch forward hooks

This is the most important engineering technique for the project.

A forward hook lets us intercept activations as they pass through the model, without changing model weights or rewriting the architecture.

### Why this matters

If we want to understand how the model behaves internally, we often need to inspect:

- hidden states at a particular layer
- attention outputs
- logits at the final prediction
- embeddings before the transformer stack

This can be done by attaching hooks to modules.

### Hook concept

In PyTorch, a hook is a callback executed when a module runs.

Example pattern:

```python
handle = module.register_forward_hook(hook_fn)
```

The hook receives:

- the module
- the input
- the output

We can store the outputs for later analysis.

### Example: capturing logits

```python
outputs = model(input_ids)
logits = outputs.logits
```

or, more directly:

```python
logits_cache = {}

def hook(module, inputs, output):
    logits_cache["logits"] = output

model.lm_head.register_forward_hook(hook)
```

### Example: capturing hidden states

```python
hidden_cache = {}

def hook(module, inputs, output):
    hidden_cache["hidden_states"] = output

layer = model.model.layers[12]
layer.register_forward_hook(hook)
```

### Why this is useful in research

This is the bridge between model internals and statistical signals:

- entropy from logits
- hidden-state norms from intermediate layers
- attention concentration patterns
- layer-wise changes during generation

This is exactly the kind of engineering skill needed for extracting interpretable signals from a black-box model without modifying the underlying weights.

### Minimal example with Hugging Face

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")

logits_store = {}

def capture_logits(module, inputs, output):
    logits_store["value"] = output

model.lm_head.register_forward_hook(capture_logits)

with torch.no_grad():
    model(**inputs)

logits = logits_store["value"]
print(logits.shape)
```

This gives access to the raw unnormalized scores over the vocabulary, which we can then convert to probabilities with softmax and use for entropy, confidence, and generation analysis.

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# This block is intentionally lightweight and conceptual.
# It shows the exact hook pattern used in real LLM analysis workflows.

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")

logits_store = {}

def capture_logits(module, inputs, output):
    logits_store["value"] = output

model.lm_head.register_forward_hook(capture_logits)

with torch.no_grad():
    model(**inputs)

logits = logits_store["value"]
print(logits.shape)
print(logits[:, -1, :].shape)

print("Forward hooks allow us to capture logits and hidden states without altering model weights.")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7110.94it/s]


torch.Size([1, 5, 50257])
torch.Size([1, 50257])
Forward hooks allow us to capture logits and hidden states without altering model weights.


## Summary

This notebook gives the mechanics behind the signals used later in the project.

At a high level:

- tokenization turns text into model-readable units
- embeddings map tokens to vectors
- attention lets tokens look at each other through context
- the model outputs logits
- softmax turns logits into probabilities
- autoregressive decoding repeats this process token by token
- forward hooks let us inspect the internals without modifying the model

This is the foundation for almost every later method in the project:

- entropy estimation
- semantic entropy
- log-probability scoring
- detection of uncertainty or hallucination risk
- hidden-state probing for model behavior analysis

The key idea is that the model is not mysterious: it is a sequence of learned transformations producing a probability distribution over the next token, one step at a time.